In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

In [33]:
OHCO = ['country_id', 'article_id','section_id','subsection_id','sent_id','token_id']

In [34]:
data_prefix = 'const'
data_dir = "parsed_data"

In [35]:
LIB = pd.read_csv(f'{data_dir}/{data_prefix}_LIB.csv')
CORPUS = pd.read_csv(f'{data_dir}/{data_prefix}_TOKEN.csv').set_index(OHCO)

In [36]:
print(pd.read_csv(f'{data_dir}/{data_prefix}_LIB.csv').columns.tolist())

['doc_id', 'country_name', 'doc_year', 'doc_len']


In [37]:
LIB = LIB.set_index('doc_id')
LIB.head()

,country_name,doc_year,doc_len
doc_id,,,
Afghanistan_2004,Afghanistan,2004,66806
Albania_2008,Albania,2008,86022
Algeria_2008,Algeria,2008,66590
Andorra_1993,Andorra,1993,55687
Angola_2010,Angola,2010,175148


In [40]:
table_dir = 'derived_tables'

In [41]:
DTM = pd.read_parquet(f'{table_dir}/const_DTM.parquet')

In [42]:
DTM.head()

term_str,0,00,000,00w,01,010,01000,01285,02,020,...,ñappointing,ñcreate,ñsuspend,ñthe,ñto,órdenes,órganos,ô,örebro,única
country_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Albania_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Algeria_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Andorra_1993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Angola_2010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
VOCAB = DTM.sum().to_frame('n')
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = np.log2(1/VOCAB.p)
VOCAB['df'] = DTM.astype(bool).sum()
VOCAB['dfidf'] = VOCAB.df * np.log2(len(DTM)/VOCAB.df)

In [44]:
TFIDF = pd.read_parquet(f'{table_dir}/const_TFIDF_L2.parquet')

In [45]:
TFIDF.head()

term_str,0,00,000,00w,01,010,01000,01285,02,020,...,ñappointing,ñcreate,ñsuspend,ñthe,ñto,órdenes,órganos,ô,örebro,única
country_id,,,,,,,,,,,,,,,,,,,,,
Afghanistan_2004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Albania_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Algeria_2008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Andorra_1993,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Angola_2010,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [46]:
VOCAB.head()

,n,p,i,df,dfidf
term_str,,,,,
0,5.0,2.920605e-06,18.385301,3,18.000000
00,6.0,3.504726e-06,18.122267,1,7.584963
000,1.0,5.841210e-07,20.707229,1,7.584963
00w,1.0,5.841210e-07,20.707229,1,7.584963
01,7.0,4.088847e-06,17.899874,3,18.000000


In [47]:
VOCAB.index

Index(['0', '00', '000', '00w', '01', '010', '01000', '01285', '02', '020',
       ...
       'ñappointing', 'ñcreate', 'ñsuspend', 'ñthe', 'ñto', 'órdenes',
       'órganos', 'ô', 'örebro', 'única'],
      dtype='object', name='term_str', length=24135)

In [48]:
fig = px.scatter(
    VOCAB,
    x='i',
    y='df',
    hover_name=VOCAB.index,  # or 'level_0'
    width=800
)

fig.show()

In [49]:
df_thresh = 50 # Remove proper nouns and one-offs
i_thresh = 10 # Remove stopwords
SIGS = VOCAB[(VOCAB.df >= df_thresh) & (VOCAB.i > i_thresh)].index
len(SIGS)


1224

In [50]:
TFIDF_SIGS = TFIDF[SIGS].copy()

In [51]:
from sklearn.decomposition import PCA

n_components = 5
pca_engine = PCA(n_components=n_components, random_state=42)
DCM = pd.DataFrame(pca_engine.fit_transform(TFIDF_SIGS), index=TFIDF_SIGS.index)
DCM

,0,1,2,3,4
country_id,,,,,
Afghanistan_2004,-0.032783,-0.004505,-0.010180,-0.012605,-0.000034
Albania_2008,-0.022837,-0.006730,-0.007345,-0.010753,-0.010947
Algeria_2008,-0.031417,-0.006499,0.000257,-0.002728,-0.001829
Andorra_1993,-0.032601,-0.007657,-0.007969,-0.011700,-0.015857
Angola_2010,-0.021704,-0.001107,-0.049123,0.014952,-0.016865
...,...,...,...,...,...
Vanuatu_1983,0.027360,0.000225,-0.009404,0.009304,0.003503
Venezuela_2009,-0.032741,-0.001276,-0.009012,-0.013822,0.003831
Yemen_2001,-0.028577,-0.004027,0.002425,-0.013969,-0.002399


In [52]:
pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_)

array([[ 4.40605776e-03,  1.42678393e-03,  7.06340729e-05,
         8.68049380e-04, -3.64676687e-04],
       [ 9.13861803e-06,  1.68888294e-03,  2.40804362e-04,
         2.45452093e-04,  3.67095069e-04],
       [ 4.67076497e-04,  7.21214216e-04,  3.11079280e-04,
         4.60773305e-05, -3.40288626e-04],
       ...,
       [ 8.48449874e-04, -1.97632491e-04,  3.55206204e-04,
         3.34934038e-04,  1.57531525e-04],
       [ 1.96695196e-04, -1.24736270e-04, -5.43169949e-05,
         2.53153877e-04,  3.99983046e-04],
       [ 1.92761914e-03, -1.38898624e-04,  1.03301048e-03,
         3.89317369e-04,  3.99399701e-04]])

In [53]:
LOADINGS = pd.DataFrame(pca_engine.components_.T * np.sqrt(pca_engine.explained_variance_), index=TFIDF_SIGS.columns)
LOADINGS.index.name = 'term_str'
LOADINGS

,0,1,2,3,4
term_str,,,,,
1,0.004406,1.426784e-03,0.000071,0.000868,-0.000365
10,0.000009,1.688883e-03,0.000241,0.000245,0.000367
11,0.000467,7.212142e-04,0.000311,0.000046,-0.000340
12,0.000444,4.946156e-04,0.000090,0.000061,-0.000279
13,0.000334,2.918695e-04,0.000202,-0.000097,-0.000029
...,...,...,...,...,...
world,-0.000544,5.288080e-07,-0.000665,0.000320,-0.000216
would,0.000909,8.944009e-05,0.000080,0.000110,0.000172
writing,0.000848,-1.976325e-04,0.000355,0.000335,0.000158


In [54]:
DOC = (
    pd.DataFrame(index=DTM.index)
    .reset_index()
    .join(LIB, on='country_id')
    .set_index(['country_id'])
)
DOC

,country_name,doc_year,doc_len
country_id,,,
Afghanistan_2004,Afghanistan,2004,66806
Albania_2008,Albania,2008,86022
Algeria_2008,Algeria,2008,66590
Andorra_1993,Andorra,1993,55687
Angola_2010,Angola,2010,175148
...,...,...,...
Vanuatu_1983,Vanuatu,1983,52183
Venezuela_2009,Venezuela,2009,240449
Yemen_2001,Yemen,2001,68130


In [55]:
LOADINGS.head()

,0,1,2,3,4
term_str,,,,,
1,0.004406,0.001427,0.000071,0.000868,-0.000365
10,0.000009,0.001689,0.000241,0.000245,0.000367
11,0.000467,0.000721,0.000311,0.000046,-0.000340
12,0.000444,0.000495,0.000090,0.000061,-0.000279
13,0.000334,0.000292,0.000202,-0.000097,-0.000029


In [56]:
def vis_pcs(x, y, color_col):
    return px.scatter(DCM.join(DOC), x, y, 
                      color=color_col, 
                      hover_name='country_name',
                      marginal_x='box', 
                      height=1000)

In [57]:
def vis_loadings(x, y):
    return px.scatter(LOADINGS.join(VOCAB), x, y, 
                      hover_name=LOADINGS.index, 
                      height=1000, 
                      size='df',
                      color='i',
                      text=LOADINGS.index)


In [58]:
vis_pcs(0, 1, 'doc_year')

In [59]:
vis_pcs(3, 4, 'doc_len')